## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../'))

from neuro_fuzzy_toolbox import ANFIS, Hybrid_learning_algorithm, EarlyStopping, get_measures

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Binary Classification

## Data

In [4]:
from ucimlrepo import fetch_ucirepo

In [5]:
parkinsons = fetch_ucirepo(id=174)

X = parkinsons.data.features
y = parkinsons.data.targets

In [6]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [7]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[ 33 103]


In [8]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[15 44]


In [9]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [10]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [11]:
train_loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 8, shuffle = True)
x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

## Model & Training

In [12]:
model = ANFIS(
    input_size = 22,
    fuzzy_rules = 15,
    outputs = 1,
    rule_reduced = True,
    output_type='binary',
    dtype=torch.float64
)

#model.init_premises(x_train)

In [13]:
loss_fn = nn.functional.binary_cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.01, 'weight_decay': 0.1}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = EarlyStopping(
    patience=50, 
    delta=0.01
)

In [14]:
trainer = Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=0.3,
    early_stopping=early_stopping
)

In [15]:
trainer(model, train_loader, verbose=True)

Epoch:   1/500 - loss: 0.516545 - validation loss: 0.528073
Epoch:   2/500 - loss: 0.529161 - validation loss: 0.566489
Epoch:   3/500 - loss: 0.496247 - validation loss: 0.483201
Epoch:   4/500 - loss: 0.491070 - validation loss: 0.477978
Epoch:   5/500 - loss: 0.509814 - validation loss: 0.529568
Epoch:   6/500 - loss: 0.502778 - validation loss: 0.518267
Epoch:   7/500 - loss: 0.499863 - validation loss: 0.512784
Epoch:   8/500 - loss: 0.505909 - validation loss: 0.528242
Epoch:   9/500 - loss: 0.504235 - validation loss: 0.509658
Epoch:  10/500 - loss: 0.505203 - validation loss: 0.499998
Epoch:  11/500 - loss: 0.508812 - validation loss: 0.516508
Epoch:  12/500 - loss: 0.504957 - validation loss: 0.525351
Epoch:  13/500 - loss: 0.498938 - validation loss: 0.529451
Epoch:  14/500 - loss: 0.496892 - validation loss: 0.508827
Epoch:  15/500 - loss: 0.504237 - validation loss: 0.511126
Epoch:  16/500 - loss: 0.509266 - validation loss: 0.532320
Epoch:  17/500 - loss: 0.488340 - valida

In [16]:
train_measures = get_measures(model, x_train, y_train)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.8676470588235294
Precision: 0.8512396694214877
Recall: 1.0
F1: 0.9196428571428571
Confusion Matrix: [[ 15  18]
 [  0 103]]


In [17]:
test_measures = get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.8305084745762712
Precision: 0.8148148148148148
Recall: 1.0
F1: 0.8979591836734694
Confusion Matrix: [[ 5 10]
 [ 0 44]]
